# Exploratory Data Analysis
Analyse the train / validation / test splits: style distribution, text lengths, vocabulary overlap, and sample quality.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from rl_rewriter.dataset import clean_dataset, load_dataset

DATA_DIR = REPO_ROOT / 'data' / 'processed'
splits = {name: clean_dataset(load_dataset(DATA_DIR / f'{name}.csv'))
          for name in ('train', 'validation', 'test')}

for name, rows in splits.items():
    print(f'{name}: {len(rows)} rows')

## Style distribution

In [ ]:
from collections import Counter

for name, rows in splits.items():
    counts = Counter(r['target_style'] for r in rows)
    print(f'{name}: {dict(counts)}')

## Text length distribution (tokens)

In [ ]:
import re

def token_count(text):
    return len(re.findall(r'\S+', text))

all_rows = [r for rows in splits.values() for r in rows]
orig_lens = [token_count(r['original_text']) for r in all_rows]
styled_lens = [token_count(r['styled_text']) for r in all_rows]

print(f'Original  — min: {min(orig_lens)}  max: {max(orig_lens)}  avg: {sum(orig_lens)/len(orig_lens):.1f}')
print(f'Styled    — min: {min(styled_lens)}  max: {max(styled_lens)}  avg: {sum(styled_lens)/len(styled_lens):.1f}')

## Per-style average text length

In [ ]:
from collections import defaultdict

style_lengths = defaultdict(list)
for r in all_rows:
    style_lengths[r['target_style']].append(token_count(r['styled_text']))

for style, lengths in sorted(style_lengths.items()):
    print(f'{style}: avg={sum(lengths)/len(lengths):.1f}  min={min(lengths)}  max={max(lengths)}')

## Vocabulary size per style

In [ ]:
style_vocab = defaultdict(set)
for r in all_rows:
    tokens = set(re.findall(r"[a-zA-Z']+", r['styled_text'].lower()))
    style_vocab[r['target_style']].update(tokens)

for style, vocab in sorted(style_vocab.items()):
    print(f'{style}: {len(vocab)} unique tokens')

## Sample records per style

In [ ]:
seen = set()
for r in all_rows:
    style = r['target_style']
    if style not in seen:
        seen.add(style)
        print(f'--- {style.upper()} ---')
        print(f'Original: {r["original_text"]}')
        print(f'Styled:   {r["styled_text"]}')
        print()

## Jaccard overlap between original and styled text per style

In [ ]:
def jaccard(a, b):
    sa, sb = set(re.findall(r"[a-zA-Z']+", a.lower())), set(re.findall(r"[a-zA-Z']+", b.lower()))
    return len(sa & sb) / len(sa | sb) if sa or sb else 0.0

overlap_by_style = defaultdict(list)
for r in all_rows:
    overlap_by_style[r['target_style']].append(jaccard(r['original_text'], r['styled_text']))

for style, overlaps in sorted(overlap_by_style.items()):
    avg = sum(overlaps) / len(overlaps)
    print(f'{style}: avg Jaccard overlap = {avg:.3f}')